# 02 · Build historical features
Pre-send features for every historical assignment. **Grain:** one row per `assignment_id`.
**Cutoff:** event ts `< send_ts`; windows are `[send_ts - N days, send_ts)`. Feature logic lives in `src/features.py`
so Step 07 can rebuild identical features at `policy_as_of_ts` (inclusive).
**Input:** Step 01 clean core tables. **Output:** `data/processed/historical_features.{csv,parquet}`, `feature_schema.csv`.

## Load Step 01 clean tables

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np, pandas as pd
from src.common import TABLES, load_clean, save
from src.features import build_features, FAMILIES, FEATURES, CATEGORICAL
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 50)
t = load_clean()
spine = t["historical_assignments"]
CONTEXT = ["assignment_id", "customer_id", "campaign_id", "send_ts", "assignment_source", "offer_type", "email_delivered"]
print(spine.shape, spine.assignment_id.is_unique)

(26600, 7) True


## Build features

In [2]:
feats, audit = build_features(spine, t, key="assignment_id", cut="send_ts", inclusive=False)
hf = spine[CONTEXT].merge(feats.drop(columns=["customer_id", "send_ts"]), on="assignment_id", how="left", validate="1:1")
print(hf.shape, "| unique assignment_id:", hf.assignment_id.is_unique)

(26600, 50) | unique assignment_id: True


## Leakage test 1 — max (event_ts − send_ts) per feature family must be < 0

In [3]:
lk = pd.Series({k: v / pd.Timedelta(hours=1) for k, v in audit.items()}, name="max_event_minus_send_hours")
print(lk.round(2).to_dict())
assert (lk < 0).all(), "post-send event used in a feature"

{'purchase': -132.77, 'purchase_refunds': -3.67, 'prior_assignments': -504.0, 'returns': -3.67, 'web': -0.47, 'email': -409.0}


## Leakage test 2 — perturbation: inject events at/after send_ts; features must not change

In [4]:
rng = np.random.default_rng(42)
samp = spine.sample(frac=1, random_state=42).drop_duplicates("customer_id").head(400)
prior_o = samp.merge(t["orders"], on="customer_id").query("order_ts < send_ts").drop_duplicates("assignment_id")
tp = {k: v.copy() for k, v in t.items()}
s_ts = samp.send_ts + pd.to_timedelta(rng.integers(0, 14 * 24, len(samp)), unit="h")
n = len(samp)
fake_o = pd.DataFrame({"order_id": [f"FAKE{i}" for i in range(n)], "customer_id": samp.customer_id.values, "order_ts": s_ts.values,
                       "campaign_id": samp.campaign_id.values, "order_source": "campaign_attributed", "order_status": "completed",
                       "offer_redeemed": 1, "offer_type_redeemed": "PCT_10", "merchandise_gross_revenue": 999.0,
                       "discount_amount": 99.9, "shipping_revenue": 0.0, "net_revenue": 899.1, "cogs": 500.0,
                       "shipping_cost": 9.0, "is_completed": True})
tp["orders"] = pd.concat([tp["orders"], fake_o], ignore_index=True)
tp["order_items"] = pd.concat([tp["order_items"], pd.DataFrame({"order_id": fake_o.order_id, "line_number": 1,
                               "product_id": "CLM_001", "quantity": 5, "unit_list_price": 199.8, "unit_cost": 100.0})])
tp["returns"] = pd.concat([tp["returns"], pd.DataFrame({"return_id": [f"RF{i}" for i in range(len(prior_o))], "order_id":
                           prior_o.order_id.values, "return_ts": prior_o.send_ts.values, "refund_amount": 50.0,
                           "return_reason": "Other"})])
tp["web_sessions"] = pd.concat([tp["web_sessions"], pd.DataFrame({"session_id": [f"SF{i}" for i in range(n)], "customer_id":
                                pd.array(samp.customer_id.values, dtype="Int64"), "session_ts": samp.send_ts.values,
                                "device_type": "mobile", "pages_viewed": 15, "added_to_cart": 1, "category_viewed": "Climbing",
                                "traffic_source": "email", "is_identified": True})])
tp["email_events"] = pd.concat([tp["email_events"], pd.DataFrame({"email_event_id": [f"EF{i}" for i in range(n)], "assignment_id":
                                samp.assignment_id.values, "customer_id": samp.customer_id.values, "campaign_id":
                                samp.campaign_id.values, "event_ts": s_ts.values, "event_type": "click"})])
f_pert, _ = build_features(samp, tp)
f_base = feats.set_index("assignment_id").loc[samp.assignment_id].reset_index()
pd.testing.assert_frame_equal(f_base[FEATURES].reset_index(drop=True), f_pert[FEATURES].reset_index(drop=True))
print("perturbation test PASS: injected post-send orders/returns/sessions/email events left all features unchanged")

perturbation test PASS: injected post-send orders/returns/sessions/email events left all features unchanged


## Leakage test 3 — current-assignment outcomes never enter features

In [5]:
first = hf[hf.campaign_id == "CP01"]
assert (first[["prior_emails_delivered", "prior_email_opens", "prior_email_clicks"]] == 0).all().all()
assert first.prior_offer_redemption_rate.isna().all()
print("CP01 rows have zero prior email history and no prior redemption history: PASS")

CP01 rows have zero prior email history and no prior redemption history: PASS


## Sanity checks

In [6]:
assert hf.assignment_id.is_unique and len(hf) == len(spine)
assert hf.customer_tenure_days.ge(0).all()
fam = pd.Series({k: len(v) for k, v in FAMILIES.items()}, name="n_features")
print(fam.to_dict(), "| total:", fam.sum())
hf[FEATURES].describe(include="all").T[["count", "mean", "min", "max"]].head(45)

{'profile': 8, 'purchase': 13, 'returns': 3, 'web': 8, 'email': 7, 'product': 4} | total: 43


,count,mean,min,max
customer_tenure_days,26600.0,632.789825,26.416667,1236.416667
is_member,26600.0,0.281316,0.0,1.0
marketing_consent,26600.0,1.0,1.0,1.0
region,26600,NaN,NaN,NaN
acquisition_channel,26371,NaN,NaN,NaN
customer_tier,26600,NaN,NaN,NaN
preferred_category,26245,NaN,NaN,NaN
shipping_zone,26600,NaN,NaN,NaN
prior_order_count,26600.0,0.36688,0.0,6.0
orders_30d,26600.0,0.047895,0.0,2.0


In [7]:
hf.sample(3, random_state=0).T

,23076,6084,5740
assignment_id,A000023077,A000006085,A000005741
customer_id,101406,107650,107890
campaign_id,CP07,CP02,CP02
send_ts,2025-05-21 10:00:00,2025-02-05 10:00:00,2025-02-05 10:00:00
assignment_source,business_rule,business_rule,business_rule
offer_type,NO_OFFER,PCT_10,PCT_10
email_delivered,1,1,1
customer_tenure_days,1103.416667,944.416667,709.416667
is_member,0,0,0
marketing_consent,1,1,1


## Save output + feature schema
Profile fields have no effective-date history; they are treated as descriptors available at decision time (documented assumption).
`products.return_rate_prior` is excluded (time provenance undocumented). `email_delivered` is context, not a feature.

In [8]:
save(hf, "historical_features")
schema = pd.DataFrame([{"feature": f, "family": k, "type": "categorical" if f in CATEGORICAL else "numeric"}
                       for k, v in FAMILIES.items() for f in v])
schema = pd.concat([pd.DataFrame({"feature": CONTEXT, "family": "context", "type": "id/context (not a predictor)"}), schema])
schema.to_csv(TABLES / "feature_schema.csv", index=False)
print("saved historical_features", hf.shape)

saved historical_features (26600, 50)
